# Gen11 Brakes — Combined Thermal Analysis

Unified Jupyter notebook for both operating cases:
- **Case A — 1g Dynamic Braking** (from `1g dynamic.py`) — transient stop $v_0 \to 0$
- **Case B — Continuous Downhill** (from `continuous downhill.py`) — steady-speed grade holding

All calculations are **front total + rear total**. Ideal distribution assumes braking force split proportional to dynamic normal loads.

Run all cells top-to-bottom after editing the shared **Vehicle Inputs** cell.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

try:
    import ipywidgets as widgets
    from IPython.display import display, Markdown
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

g = 9.81
print("imports ok — g =", g)

## Shared Vehicle Inputs (edit once, used by both cases)

Defaults from `Brake_Calcs_Front_Rear_1g_Max_Braking.ipynb`: $w=272\,kg, wb=2.3\,m, z_{cog}=19.90\,$in.

In [ ]:
# ── SHARED VEHICLE ──
total_mass = 272.0              # kg
static_mass_front = 177.64      # kg
static_mass_rear = 94.36        # kg  (front+rear must = total)
axle_distance = 2.3             # m wheelbase L
cog_height = 19.90 * 0.0254     # m CoG height h

# ── CASE A: 1g dynamic ──
initial_speed = 30.0   # m/s
final_speed = 0.0      # m/s
decel = 9.81           # m/s^2

# ── CASE B: continuous downhill ──
constant_speed = 25.0       # m/s steady
downhill_angle = 5.0        # degrees
downhill_distance = 8000.0  # m along slope

# derived geometry (shared)
total_weight = total_mass * g
front_to_cog = axle_distance * static_mass_rear / total_mass
rear_to_cog  = axle_distance - front_to_cog
print(f"Vehicle: {total_mass} kg  L={axle_distance}m  h={cog_height:.4f}m  front_to_cog={front_to_cog:.3f}m rear_to_cog={rear_to_cog:.3f}m")
print(f"Case A: {initial_speed}→{final_speed} m/s @ {decel/g:.2f}g")
print(f"Case B: {constant_speed} m/s @ {downhill_angle}° for {downhill_distance/1000:.1f} km")

## Case A — 1g Dynamic Braking

$F_1 = (W\cdot b_{rear} + m a h)/L$

In [ ]:
energy_change = 0.5*total_mass*(initial_speed**2 - final_speed**2)
braking_force_total_A = total_mass * decel

N1_A = (total_weight*rear_to_cog + total_mass*decel*cog_height)/axle_distance
N2_A = total_weight - N1_A
B1_A = braking_force_total_A * N1_A/total_weight
B2_A = braking_force_total_A - B1_A
E1_A = energy_change * N1_A/total_weight
E2_A = energy_change - E1_A

print(f"N1={N1_A:,.0f} N ({N1_A/total_weight*100:.1f}%)  N2={N2_A:,.0f} N ({N2_A/total_weight*100:.1f}%)")
print(f"B1={B1_A:,.0f} N  B2={B2_A:,.0f} N  (total {braking_force_total_A:,.0f} N)")
print(f"Energy: front {E1_A/1000:.1f} kJ ({E1_A/energy_change*100:.1f}%)  rear {E2_A/1000:.1f} kJ  total {energy_change/1000:.1f} kJ")
print(f"Per rotor: front {E1_A/2/1000:.1f} kJ  rear {E2_A/2/1000:.1f} kJ")
t_stop = (initial_speed-final_speed)/decel
d_stop = (initial_speed**2-final_speed**2)/(2*decel)
print(f"t_stop={t_stop:.2f}s  d_stop={d_stop:.1f}m  avg power={energy_change/t_stop/1000:.1f} kW")

## Case B — Continuous Downhill (steady speed)

$N_1=(W\cos\theta\cdot b + W\sin\theta\cdot h)/L$

In [ ]:
th = math.radians(downhill_angle)
Wn = total_weight*math.cos(th)
Wp = total_weight*math.sin(th)
Btot_B = Wp
Ptot_B = Btot_B * constant_speed

N1_B = (total_weight*math.cos(th)*rear_to_cog + total_weight*math.sin(th)*cog_height)/axle_distance
N2_B = Wn - N1_B
B1_B = Btot_B * N1_B / Wn
B2_B = Btot_B - B1_B
P1_B = Ptot_B * N1_B / Wn
P2_B = Ptot_B - P1_B
t_desc = downhill_distance/constant_speed
Etot_B = Btot_B * downhill_distance
E1_B = Etot_B * N1_B / Wn
E2_B = Etot_B - E1_B

print(f"Wn={Wn:,.0f} N  Wp={Wp:,.0f} N  (theta={downhill_angle}°)")
print(f"N1={N1_B:,.0f} N ({N1_B/Wn*100:.1f}%)  N2={N2_B:,.0f} N")
print(f"B1={B1_B:,.0f} N ({B1_B/Btot_B*100:.1f}%)  B2={B2_B:,.0f} N  total {Btot_B:,.0f} N")
print(f"Power: front {P1_B:,.0f} W  rear {P2_B:,.0f} W  total {Ptot_B:,.0f} W ({Ptot_B/1000:.2f} kW)")
print(f"Per rotor power: front {P1_B/2:,.0f} W  rear {P2_B/2:,.0f} W")
print(f"Time {t_desc/60:.1f} min  Energy: front {E1_B/1e6:.2f} MJ  rear {E2_B/1e6:.2f} MJ  total {Etot_B/1e6:.2f} MJ")

## Comparison Plots

In [ ]:
labels = ['Front', 'Rear']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Row 1: Case A
axes[0,0].bar(labels, [N1_A, N2_A], color=['#1f77b4','#ff7f0e'])
axes[0,0].set_title('A — Normal Load (N)')
axes[0,1].bar(labels, [B1_A, B2_A], color=['#1f77b4','#ff7f0e'])
axes[0,1].set_title('A — Braking Force (N)')
axes[0,2].bar(labels, [E1_A/1000, E2_A/1000], color=['#1f77b4','#ff7f0e'])
axes[0,2].set_title('A — Energy (kJ)')
for ax in axes[0]:
    for c in ax.containers:
        ax.bar_label(c, fmt='%.0f', padding=3, fontsize=8)

# Row 2: Case B
axes[1,0].bar(labels, [N1_B, N2_B], color=['#2ca02c','#d62728'])
axes[1,0].set_title(f'B — Normal Load on {downhill_angle}° (N)')
axes[1,1].bar(labels, [P1_B, P2_B], color=['#2ca02c','#d62728'])
axes[1,1].set_title('B — Power (W)')
axes[1,2].bar(labels, [E1_B/1e6, E2_B/1e6], color=['#2ca02c','#d62728'])
axes[1,2].set_title(f"B — Energy over {downhill_distance/1000:.0f}km (MJ)")
for ax in axes[1]:
    for c in ax.containers:
        ax.bar_label(c, fmt='%.2f', padding=3, fontsize=8)

fig.suptitle('Gen11 Brakes — Case A (1g stop) vs Case B (steady downhill)', fontsize=12)
plt.tight_layout()
plt.show()

print(f"A front bias: {N1_A/total_weight*100:.1f}%  |  B front bias: {N1_B/Wn*100:.1f}%")
print(f"A avg power: {energy_change/t_stop/1000:.1f} kW for {t_stop:.1f}s  |  B continuous: {Ptot_B/1000:.2f} kW for {t_desc/60:.1f} min")

In [ ]:
# summary table
import pandas as pd
df = pd.DataFrame({
    'Case': ['1g stop (total)', '1g front', '1g rear', 'Downhill total', 'Downhill front', 'Downhill rear'],
    'Normal (N)': [total_weight, N1_A, N2_A, Wn, N1_B, N2_B],
    'Brake Force (N)': [braking_force_total_A, B1_A, B2_A, Btot_B, B1_B, B2_B],
    'Power (W)': [energy_change/t_stop, energy_change/t_stop * N1_A/total_weight, energy_change/t_stop * N2_A/total_weight, Ptot_B, P1_B, P2_B],
    'Energy': [f"{energy_change/1e3:.1f} kJ", f"{E1_A/1e3:.1f} kJ", f"{E2_A/1e3:.1f} kJ", f"{Etot_B/1e6:.2f} MJ", f"{E1_B/1e6:.2f} MJ", f"{E2_B/1e6:.2f} MJ"],
})
df